# Obtaining systemic redshifts from stacked emission

This script estimates the systemic redshift of each source by stacking optically-thin emission lines and fitting gaussian profiles

In [ ]:
import numpy as np
from astropy.io import fits
import astropy.table as aptb
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

import copy

from tangelo import spectroscopy as tgspec
from tangelo import models as tgmod
from tangelo import fitting as tgfit
from tangelo import catalogue_operations as tgcat
from tangelo import constants as tgconst
from tangelo import io as tgio

In [ ]:
# Load the megatab
SPEC_SOURCE = "R21"  #  Where are the spectra from? 'R21' for Richard+21, 'APER' for apertures
SPEC_TYPE   = "weight_skysub" # Which subtype of spectrum to use (e.g. 'Nfwhm_opt' for APER, 'weight_skysub' for R21)   

tabfile = f"megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_{SPEC_TYPE}.fits"
tabhdul = fits.open(tabfile)
megatable = aptb.Table(tabhdul[1].data)

In [ ]:
# Get the dictionary of lines to look for
wavedict = tgconst.wavedict

# Clean the megatable to only include spectra with good SNR in red or blue Lya peak
megatab = megatable[(megatable['SNRR'] > 5.0) + (megatable['SNRB'] > 5.0)]

In [ ]:
# First try to get as many measurements of sys vel as possible
# Try to re-do the Z_sys calculation for all sources, this time by averaging all the optically thin lines
# (NOT CIV) and fitting gaussians

# Fitting configuration dictionary
fit_config = {
    'sig_level'   : 3, # What significance level to accept a fit at
    'search_width': 700, # Half width of the region around the Lyman alpha peak to search for stacked emission
    'fit_width'   : 2500, # Half width of the fitting region around Lya peak
    'vel_bin'     : 'auto'
}



# These are the lines that, if already detected at >3 sigma, can be used to make the stack
good_lines =  ['CIII1907', 'CIII1909', 'HeII1640', 'OIII1660', 'OIII1666', 'SiIII1883', 'SiIII1892']

# These are the default optically thin lines to use if no good lines are detected
optthin_lines = ['HeII1640','OIII1666','CIII1907','CIII1909']

megatab_newsysz = copy.deepcopy(megatab) # make a copy to add new columns to
# Add new columns to table
megatab_newsysz.add_columns([aptb.Column(np.nan*np.zeros(len(megatab)), "DELTAV_LYA"),
                            aptb.Column(np.nan*np.zeros(len(megatab)), "DELTAV_LYA_ERR"),
                            aptb.Column(np.nan*np.zeros(len(megatab)), "VFWHM_SYS"),
                            aptb.Column(np.nan*np.zeros(len(megatab)), "VFLUX_SYS"),
                             aptb.Column(np.nan*np.zeros(len(megatab)), "VFLUX_SYS_ERR"),
                            aptb.Column(np.nan*np.zeros(len(megatab)), "VCONT_SYS"),
                            aptb.Column(np.nan*np.zeros(len(megatab)), "VSLOPE_SYS")])

hi_abslines = ['SiIV1394','SiIV1403'] # Abs lines to be plotted to compare with the sys z
li_abslines =  ['SiII1260','CII1334','SiII1527'] # Abs lines to be plotted to compare with the sys z

init_snr_thresh = 2.0

det_lines = [] # Initialise list of detected lines across all sources
count = 0 # Initialise count of significant zsys measurements
emitters = 0 # Initialise count of statistically significant emitters
for i, row in enumerate(megatab_newsysz):
    cluster = row['CLUSTER']
    iden = row['iden']
    
    print("\n" + "=" * 80)
    print(f"\nProcessing {iden} from {cluster}...")
    
    # Check for tentative unflagged emission lines found in previous step
    tentem = tgcat.is_true_emitter(row, {x: wavedict[x] for x in good_lines}, n=3, sig=3, return_lines=True)
    verified_emitter = False
    if tentem[0]:
        emlines = list(
            set(tentem[1][0])
            & set(good_lines)
        )
        print(f"Using the following lines detected at > 3σ: {emlines}")
        emitters += len(emlines) > 0 # counting number of verified optically thin emitters
        det_lines.extend(emlines) # Add them to the list of detected lines
        if len(emlines) < 3:
            print('Insufficient optically thin lines. Reverting to default line list...')
            emlines = optthin_lines
        else:
            verified_emitter = True
    else:
        print(f"Insufficient emission lines found. Using default line list.")
        emlines = optthin_lines
    
    # Get the average line profile as well as Lyman alpha and absorption lines for comparison
    z_lya = row['LPEAKR'] / 1215.67 - 1.
    velbounds = [-2500, 2500]
    vel, spec, specerr = tgspec.avg_lines(row, emlines, absorption=False, velbounds=velbounds, z = z_lya, flags=['c','s','d'],
                                          spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE, velstep=fit_config['vel_bin'],
                                         use_measured_var=False)
    lyvel, lyspec, lyspecerr = tgspec.avg_lines(row, ['LYALPHA'], absorption=False, velbounds=velbounds, z = z_lya, 
                                                spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE, velstep=fit_config['vel_bin'])
    abvel, abspec, abspecerr = tgspec.avg_lines(row, li_abslines, absorption=False, velbounds=velbounds, z = z_lya, 
                                                spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE, velstep=fit_config['vel_bin'],
                                                use_measured_var=True)
    habvel, habspec, habspecerr = tgspec.avg_lines(row, hi_abslines, absorption=False, velbounds=velbounds, z = z_lya, 
                                                   spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE, velstep=fit_config['vel_bin'],
                                                   use_measured_var=True)
    
    # Check for NaNs in the averaged line profile, which will indicate no lines were found
    if np.sum(np.isnan(vel * spec * specerr)) > 0:
        print("Source redshift is too high, no lines to fit. Skipping.\n")
        continue

    # Plot the line profiles for visual inspection
    fig, ax = plt.subplots(4, 1, figsize=(5,5), facecolor='white', sharex=True)
    fig.subplots_adjust(hspace=0)
    ax[0].set_title(f"{row['CLUSTER']} ID {row['iden']}: $ z = {row['z']:.4f} $")
    ax[1].set_xlabel(r'$v$ [km\,s$^{-1}$]')
    ax[1].set_ylabel(r'$f_v$ [$10^{-20}$\,erg\,s$^{-1}$\,cm$^{-2}$\,(km\,s$^{-1}$)$^{-1}$]')
    ax[0].plot(lyvel, lyspec, drawstyle='steps-mid', color='tab:red')
    ax[0].fill_between(lyvel, lyspec - lyspecerr, lyspec + lyspecerr, step='mid', facecolor='tab:red', alpha=0.3, linestyle='')
    ax[1].plot(vel, spec, drawstyle='steps-mid', color='tab:orange')
    ax[1].fill_between(vel, spec - specerr, spec + specerr, step='mid', facecolor='tab:orange', alpha=0.3, linestyle='')
    ax[2].plot(abvel, abspec, drawstyle='steps-mid', color='tab:blue')
    ax[2].fill_between(abvel, abspec - abspecerr, abspec + abspecerr, step='mid', facecolor='tab:blue', alpha=0.3, linestyle='')
    ax[3].plot(habvel, habspec, drawstyle='steps-mid', color='tab:purple')
    ax[3].fill_between(habvel, habspec - habspecerr, habspec + habspecerr, step='mid', facecolor='tab:purple', alpha=0.3, linestyle='')
    
    # Now fit a gaussian to the average line profile
    fitreg = (-fit_config['fit_width'] < vel) * (vel < fit_config['fit_width'])
    fitreg &= ~(spec * specerr == 0) # some spectra have zero errors which will cause infinite normalised residuals in curve_fit
    fitreg &= ~np.isinf(spec)   # FIX 3: exclude Inf channels not caught by the zero-product check
    fitreg &= ~np.isinf(specerr)
    # Use a range of initial guesses to try to ensure the fitter finds the global minimum, not a local one
    search_reg = (-fit_config['search_width'] < vel) & (vel < fit_config['search_width'])
    max_idx = np.argmax(spec[search_reg])
    peak_vel = vel[search_reg][max_idx]
    vpeak_guesses = [peak_vel] # first guess is on the highest point within the search bounds
    vpeak_guesses.extend([-300, 300]) # subsequent generic guesses
    initial_guess = {
        'FLUX': (np.nanmax(spec)-np.nanmedian(spec)) * np.min(np.ediff1d(vel)),
        'VPEAK': 0,
        'FWHM': 100,
        'CONT': np.nanmedian(spec),
        'SLOPE': 0
        }
    initial_guess_series = {vpeak : {**initial_guess, 'VPEAK': vpeak} for vpeak in vpeak_guesses}
    # Perform an initial fit to get starting parameters for MCMC and judge
    # whether it is worth proceeding with the MCMC fitting (which is more computationally expensive)
    
    print("Performing initial fits to judge significance of line detection...")
    
    initial_fits = {}
    best_vpeak = None
    attempts = 3
    for j, (vpeak, guesses) in enumerate(initial_guess_series.items()):
        print(f"Fitting with initial VPEAK = {vpeak:.2f}")
        popt, pcov = np.zeros(len(guesses)), np.ones((len(guesses), len(guesses)))
        for n in range(attempts):
            try:
                popt, pcov = curve_fit(tgmod.gaussian, vel[fitreg], spec[fitreg], sigma=specerr[fitreg],
                                        p0 = list(guesses.values()), absolute_sigma=True, max_nfev=100000,
                                        bounds = [[0, -fit_config['search_width'], 25, -50, -1],
                                                [500, fit_config['search_width'], 200, 1000, 1]], method='trf')
                print(f"Fitted peak at v = {popt[1]:.2f} with SNR = {popt[0] / np.sqrt(pcov[0,0])}")
                break
            except (RuntimeError, ValueError):  # FIX 1: also catch ValueError (raised when p0 violates bounds)
                print(f"Fitting attempt {n+1} failed due to Runtime Error.")
                if n+1 < attempts:
                    print(f"Trying again...")
                    continue
                else:
                    print(f"All fit attempts failed. Moving to next guess...")
                    pass
            
        initial_fits[vpeak] = (popt, pcov)
        snr = popt[0] / np.sqrt(pcov[0][0])
        # If the first peak we land on (the highest point) is highly significant, stop the search
        if j == 0 and snr > 3:
            best_vpeak = vpeak
            break
        elif best_vpeak is None or snr > initial_fits[best_vpeak][0][0] / np.sqrt(initial_fits[best_vpeak][1][0][0]):
            # If the highest point wasn't significant, look elsewhere
            best_vpeak = vpeak

    best_snr = initial_fits[best_vpeak][0][0] / np.sqrt(initial_fits[best_vpeak][1][0][0])
    if best_snr < init_snr_thresh:
        print(f"Initial fit < {init_snr_thresh:.1f}σ. Moving to next source.\n")  # FIX 2: threshold is 1σ, not 2σ
        plt.show()
        plt.close()
        continue
    else:
        print(f"Initial fit > {init_snr_thresh:.1f}σ, proceeding with MCMC fitting...")  # FIX 2: threshold is 1σ, not 2σ

    # Determine which initial guess gives the better fit and use that as the starting point for MCMC
    best_guesses = initial_fits[best_vpeak][0]
    guesses_dict = dict(zip(initial_guess.keys(), best_guesses))
    bounds_dict = {
        'FLUX': (0,1000),
        'VPEAK': (-fit_config['search_width'], fit_config['search_width']),
        'FWHM': (25, 300),
        'CONT': (-np.inf, np.inf),
        'SLOPE': (-np.inf, np.inf)
    }

    # Build MF exclusion regions for doublet ghost features in the stacked spectrum.
    # When two lines of a doublet are both present in the stack, each appears as a ghost
    # in the other's velocity frame at ±Δv from the fitted VPEAK, where Δv = c*(λ2-λ1)/λ1.
    # We exclude a ±FWHM window around each ghost position from the MF background distribution.
    mf_exclude_regions = []
    fitted_vpeak = guesses_dict['VPEAK']
    fitted_fwhm  = guesses_dict['FWHM']
    excl_region_width = fitted_fwhm
    for primary, (_, secondary) in tgconst.doublets.items():
        if primary in emlines and secondary in emlines:
            lam1 = tgconst.wavedict[primary]
            lam2 = tgconst.wavedict[secondary]
            dv = tgconst.c * (lam2 - lam1) / lam1  # velocity offset between doublet partners
            for ghost_vel in [fitted_vpeak + dv, fitted_vpeak - dv]:
                mf_exclude_regions.append((ghost_vel - excl_region_width / 2, ghost_vel + excl_region_width / 2))
                ax[1].axvspan(ghost_vel - excl_region_width / 2, ghost_vel + excl_region_width / 2, alpha=0.3, color='grey')
        elif secondary in emlines: # this covers the case of OIII1666
            lam1 = tgconst.wavedict[primary]
            lam2 = tgconst.wavedict[secondary]
            dv = tgconst.c * (lam2 - lam1) / lam1  # velocity offset between doublet partners
            for ghost_vel in [fitted_vpeak - dv]:
                mf_exclude_regions.append((ghost_vel - excl_region_width / 2, ghost_vel + excl_region_width / 2))
                ax[1].axvspan(ghost_vel - excl_region_width / 2, ghost_vel + excl_region_width / 2, alpha=0.3, color='grey')

    print(f"Fitting with VPEAK initial guess = {guesses_dict['VPEAK']}...")
    bootstrap_params = {
        'niter': 1000,
        'max_nfev': 10000,
        'autocorrelation': True,
        'errfunc': 'stddev_7',
        'baseline_order': 1
    }
    fit_result_dict = tgfit.fit_line(vel[fitreg], spec[fitreg], specerr[fitreg], "STACK",
                                     guesses_dict, bounds = bounds_dict, continuum_buffer=2000,
                                     plot_result=False, bootstrap_params=bootstrap_params,
                                     mf_exclude_regions=mf_exclude_regions if mf_exclude_regions else None)

    param_dict = fit_result_dict['param_dict']
    error_dict = fit_result_dict['error_dict']
    popt_mc = [param_dict[key] for key in initial_guess.keys()]
    perr_mc = [error_dict[key] for key in initial_guess.keys()]
    flagged = ('m' in fit_result_dict['flags'][0]) and not verified_emitter
    if fit_result_dict['flags'][0] and verified_emitter:
        print(f"Flags were removed due to source being verified emitter.")
    significant = popt_mc[0]/perr_mc[0] > fit_config['sig_level']
    # # This condition is designed to catch cases where there are suspiciously high continuum levels
    # # in a source that allegedly had no HST continuum detection (which suggests contamination rather than
    # # genuinely high continuum)
    # continuum_anomaly_flag = popt_mc[-2] > 0.5 and row['idfrom'] == 'MUSELET'
    # The redshift of optically thin lines should not be significantly higher than that of any CIV lines (if detected at high significance)
    # THIS ACTUALLY DOESN'T WORK (ASSUMES OUTFLOWS EFFECTIVELY, SO IS DISABLED)
    # CIV_anomaly_flag = False
    # CIV_detected = (row["SNR_CIV1548"] > 5) | (row["SNR_CIV1551"] > 5)
    # if CIV_detected:
    #     # Calculate the velocity offset of the CIV line from Lyman alpha
    #     CIV_vel_offset = tgspec.wave2vel(row['LPEAK_CIV1548'], tgconst.wavedict['CIV1548'], redshift=row['LPEAKR'] / 1215.67 - 1)
    #     if CIV_vel_offset < popt_mc[1] - 3 * perr_mc[1]:
    #         CIV_anomaly_flag = True

    bad_fit = any([
        not significant, 
        flagged, 
        # CIV_anomaly_flag
    ])
    fit_color = 'grey' if bad_fit else 'fuchsia'

    velhires = np.linspace(vel[0], vel[-1], 1000)
    ax[1].plot(velhires, tgmod.gaussian(velhires, *popt_mc), linestyle='--', color=fit_color, alpha=0.6,
                    label='model')
    for a in ax:
        a.axvline(popt_mc[1], linestyle='--', color='black', alpha=0.4)
    plot_dir = tgio.get_plot_dir(cluster, iden)
    plt.savefig(plot_dir / f"{cluster}_ID{iden}_sysz_fit.png", dpi=300)
    plt.show()
    plt.close()

    if not bad_fit:
        print(f"Found a significant line detection with VPEAK = {popt_mc[1]:.1f} km/s and FWHM = {popt_mc[2]:.1f} km/s.")
    elif not significant:
        print(f"Fit is not statistically significant at the {fit_config['sig_level']}σ level. Moving to next source.\n")
    # elif continuum_anomaly_flag:
    #     print(f"Fit is flagged as having an anomalously high continuum level, which may indicate contamination. Moving to next source.\n")
    # elif CIV_anomaly_flag:
    #     print(f"Fit is flagged due to secure detection of CIV at significantly lower redshift than optically thin lines.")
    elif flagged:
        print(f"Fit was flagged by matched filtering with flag {fit_result_dict['flags'][0]}.\n")
        
    
    count += int(significant)
    print(str(count)+'/'+str(i+1))

    if bad_fit:
        continue
    
    print(f"Significant emission found. Entering sys z info into table...")
    megatab_newsysz['DELTAV_LYA'][i] = -popt_mc[1]
    megatab_newsysz['DELTAV_LYA_ERR'][i] = np.abs(perr_mc[1])
    megatab_newsysz['VFWHM_SYS'][i] = popt_mc[2]
    megatab_newsysz['VFLUX_SYS'][i] = popt_mc[0]
    megatab_newsysz['VFLUX_SYS_ERR'][i] = perr_mc[0]
    megatab_newsysz['VCONT_SYS'][i] = popt_mc[-2]
    megatab_newsysz['VSLOPE_SYS'][i] = popt_mc[-1]
    print(f"Done.\n")


In [ ]:
# Save the new megatable with systemic redshift info
megatab_newsysz.write(f"./megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_sysv_{SPEC_TYPE}.fits", overwrite=True)

In [ ]:
# print the number of emitters
print(f"Number of verified emitters with 3+ optically thin lines detected at > 3σ: {emitters}")

In [ ]:
# Check how many Lya velocity offsets are negative

megatab_goodsys = megatab_newsysz[megatab_newsysz['VFLUX_SYS'] / megatab_newsysz['VFLUX_SYS_ERR'] > 5.0]

N_neg = (megatab_goodsys['DELTAV_LYA'] < 0).sum()
N_pos = (megatab_goodsys['DELTAV_LYA'] > 0).sum()
f_neg = N_neg / (N_neg + N_pos)

print(f"Number (fraction) of negative Lyα velocity offsets:")
print(f"{N_neg} out of {N_neg + N_pos} ({f_neg:.2f})")

In [ ]:
x_vals = megatab_newsysz['VFWHM_SYS']
x_vals_good = megatab_goodsys['VFWHM_SYS']
plt.scatter(x_vals, megatab_newsysz['DELTAV_LYA'], color='grey')
plt.errorbar(x_vals, megatab_newsysz['DELTAV_LYA'], yerr=megatab_newsysz['DELTAV_LYA_ERR'], linestyle='', color='grey')
plt.scatter(x_vals_good, megatab_goodsys['DELTAV_LYA'], color='green')
plt.errorbar(x_vals_good, megatab_goodsys['DELTAV_LYA'], yerr=megatab_goodsys['DELTAV_LYA_ERR'], linestyle='', color='green')
plt.axhline(0, linestyle='--')
plt.show()
plt.close()
plt.hist(megatab_newsysz['DELTAV_LYA'], bins=10)
plt.hist(megatab_goodsys['DELTAV_LYA'], bins=10)
plt.show()